# Train Intent Model
This notebook trains a Text-to-Intent model using scikit-learn. The model takes a Vietnamese sentence and predicts the intended control command (e.g. `led:on`).
**Version 2**: Tăng dung lượng dữ liệu và nâng cấp model để nhận diện tốt hơn.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
import joblib
import os

In [2]:
# Define augmented dataset
data = {
    'text': [
        # LED ON
        'bật đèn', 'mở đèn', 'làm ơn bật đèn lên', 'bật đèn phòng khách', 'trời tối quá bật đèn đi',
        'mở điện', 'bật điện lên', 'cho sáng lên coi', 'tối thui bật đèn lên', 'bật bóng đèn', 'bật đèn ngủ',
        'sáng lên nào', 'mở đèn trần', 'bật đèn ban công', 'anh ơi bật đèn', 'chị ơi mở đèn giúp',
        
        # LED OFF
        'tắt đèn', 'đóng đèn', 'tắt cái đèn đi', 'tắt đèn giúp tôi', 'sáng quá tắt đèn đi',
        'tắt điện đi', 'tắt bóng đèn', 'tắt đèn ngủ', 'chói mắt quá tắt đèn', 'tắt giùm cái đèn',
        'tắt đèn trần đi', 'đi ngủ tắt đèn', 'không cần sáng nữa tắt đèn',
        
        # FAN ON
        'bật quạt', 'mở quạt', 'nóng quá mở quạt', 'bật quạt lên', 'mở quạt số to',
        'oi bức quá bật quạt', 'mở quạt trần', 'bật cái quạt', 'bật quạt treo tường', 'nực quá mở quạt lên',
        'cho mát chút đi bật quạt', 'anh ơi bật quạt', 'mở quạt số 3',
        
        # FAN OFF
        'tắt quạt', 'đóng quạt', 'tắt quạt đi', 'lạnh quá tắt quạt', 'làm ơn tắt quạt',
        'tắt cái quạt', 'lạnh rứa tắt quạt đi', 'tắt quạt trần', 'gió to quá tắt quạt', 'tắt quạt số 3 đi',
        'rét quá tắt quạt', 'thôi đừng quạt nữa tắt đi',
        
        # SERVO ON
        'mở cửa', 'bật rèm', 'mở servo', 'mở cửa ra', 'mở cổng đi',
        'kéo rèm ra', 'mở cửa chính', 'hé cửa ra', 'kéo màn lên', 'mở cửa sổ',
        'cho nắng vào phòng kéo rèm ra', 'mở cửa đón khách',
        
        # SERVO OFF
        'đóng cửa', 'tắt rèm', 'đóng servo', 'đóng cửa lại', 'khép cửa vào',
        'kéo rèm lại', 'đóng cổng', 'khép cái cửa', 'kéo màn xuống', 'đóng cửa sổ',
        'kéo rèm che nắng', 'đóng chặt cửa lại'
    ],
    'intent': [
        # LED ON
        'led:on', 'led:on', 'led:on', 'led:on', 'led:on',
        'led:on', 'led:on', 'led:on', 'led:on', 'led:on', 'led:on',
        'led:on', 'led:on', 'led:on', 'led:on', 'led:on',
        
        # LED OFF
        'led:off', 'led:off', 'led:off', 'led:off', 'led:off',
        'led:off', 'led:off', 'led:off', 'led:off', 'led:off',
        'led:off', 'led:off', 'led:off',
        
        # FAN ON
        'fan:on', 'fan:on', 'fan:on', 'fan:on', 'fan:on',
        'fan:on', 'fan:on', 'fan:on', 'fan:on', 'fan:on',
        'fan:on', 'fan:on', 'fan:on',
        
        # FAN OFF
        'fan:off', 'fan:off', 'fan:off', 'fan:off', 'fan:off',
        'fan:off', 'fan:off', 'fan:off', 'fan:off', 'fan:off',
        'fan:off', 'fan:off',
        
        # SERVO ON
        'servo:on', 'servo:on', 'servo:on', 'servo:on', 'servo:on',
        'servo:on', 'servo:on', 'servo:on', 'servo:on', 'servo:on',
        'servo:on', 'servo:on',
        
        # SERVO OFF
        'servo:off', 'servo:off', 'servo:off', 'servo:off', 'servo:off',
        'servo:off', 'servo:off', 'servo:off', 'servo:off', 'servo:off',
        'servo:off', 'servo:off'
    ]
}
df = pd.DataFrame(data)
print(f"Total training samples: {len(df)}")
df.sample(5)

Total training samples: 78


,text,intent
30,mở quạt,fan:on
76,kéo rèm che nắng,servo:off
33,mở quạt số to,fan:on
15,chị ơi mở đèn giúp,led:on
16,tắt đèn,led:off


In [3]:
# Create and train pipeline
# Nâng cấp: Dùng ngram_range=(1,3) để bắt cụm từ (ví dụ: 'bật đèn', 'tắt quạt')
# Dùng C=1.5 để tăng mức phạt khi model đoán sai (giúp model khắt khe hơn)
model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 3)), 
    SVC(probability=True, kernel='linear', C=1.5)
)
model.fit(df['text'], df['intent'])
print('Training complete!')

Training complete!


In [4]:
# Test model
test_sentences = [
    'anh ơi mở cái quạt trần lên', 
    'tối thui rồi bật đèn lên giùm', 
    'gió lạnh quá tắt quạt đi', 
    'nhà tối quá bật điện giúp em',
    'mở giùm cái cửa đón nắng',
    'đóng cửa sổ lại'
]
predictions = model.predict(test_sentences)
probabilities = model.predict_proba(test_sentences)

for text, intent, prob in zip(test_sentences, predictions, probabilities.max(axis=1)):
    print(f'{text} -> {intent} (Confidence: {prob:.2f})')

anh ơi mở cái quạt trần lên -> fan:on (Confidence: 0.66)
tối thui rồi bật đèn lên giùm -> led:on (Confidence: 0.95)
gió lạnh quá tắt quạt đi -> fan:off (Confidence: 0.95)
nhà tối quá bật điện giúp em -> led:on (Confidence: 0.91)
mở giùm cái cửa đón nắng -> servo:on (Confidence: 0.43)
đóng cửa sổ lại -> servo:off (Confidence: 0.90)


In [5]:
# Save model to models directory
model_dir = '../GateWay/Voice/models'
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'intent_model.pkl')
joblib.dump(model, model_path)
print(f'Model saved to {model_path}')

Model saved to ../GateWay/Voice/models\intent_model.pkl
